In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as sunpycm
from sunpy.visualization.colormaps import color_tables
from omegaconf import OmegaConf

from sdofmv2.core import SDOMLDataModule, MAE
from sdofmv2.tasks.missing_data import MissingDataModel

In [ ]:
cfg = OmegaConf.load("../../../../configs/downstream/missing_channel_sdofmv2_ALL.yaml")

data_module = SDOMLDataModule(
    hmi_path=(
        os.path.join(
            cfg.data.sdoml.base_directory,
            cfg.data.sdoml.sub_directory.hmi,
        )
        if cfg.data.sdoml.sub_directory.hmi
        else None
    ),
    aia_path=(
        os.path.join(
            cfg.data.sdoml.base_directory,
            cfg.data.sdoml.sub_directory.aia,
        )
        if cfg.data.sdoml.sub_directory.aia
        else None
    ),
    eve_path=None,
    components=cfg.data.sdoml.components,
    wavelengths=cfg.data.sdoml.wavelengths,
    ions=cfg.data.sdoml.ions,
    batch_size=cfg.model.misc.batch_size,
    num_workers=cfg.data.num_workers,
    pin_memory=cfg.data.pin_memory,
    persistent_workers=cfg.data.persistent_workers,
    multiprocessing_context=cfg.data.multiprocessing_context,
    normalization=cfg.data.sdoml.normalization,
    normalization_stat_path=cfg.data.normalization_stat_path,
    train_index=cfg.data.train_index,
    val_index=cfg.data.val_index,
    test_index=cfg.data.test_index,
    hmi_mask=cfg.data.hmi_mask,
    apply_mask=cfg.data.sdoml.apply_mask,
    num_frames=cfg.data.num_frames,
    drop_frame_dim=cfg.data.drop_frame_dim,
    precision=cfg.experiment.precision,
)
data_module.setup()
channels = data_module.wavelengths + data_module.components

In [ ]:
# Load hyper-parameters from config
config_hparams = {
    **cfg.model.mae,
    "chan_types": channels,
    "limb_mask": torch.Tensor(np.load(cfg.data.hmi_mask)),
    "loss_dict": cfg.model.loss,
    "optimizer_dict": cfg.model.optimizer,
    "scheduler_dict": cfg.model.scheduler,
}

# Load the checkpoint and its parameters
ckpt_path = os.path.join(cfg.experiment.backbone.ckpt_dir, cfg.experiment.backbone.weight_name)
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
ckpt_hparams = ckpt.get("hyper_parameters", {})

# Merge them: Checkpoint overwrites Config for matching keys
merged_hparams = {**config_hparams, **ckpt_hparams}
backbone = MAE(**merged_hparams)
zero_shot = MAE(**merged_hparams)
backbone.load_state_dict(ckpt["state_dict"], strict=False)
zero_shot.load_state_dict(ckpt["state_dict"], strict=False)

# Load finetuned model
model = MissingDataModel.load_from_checkpoint(
    checkpoint_path=os.path.join(cfg.experiment.ds_ckpt_dir, cfg.experiment.checkpoint_filename),
    map_location="cpu",
    weights_only=False,
    backbone=backbone,
)

In [ ]:
cms_dict = {
    "131a": sunpycm.cmlist.get("sdoaia131"),
    "1600a": sunpycm.cmlist.get("sdoaia1600"),
    "1700a": sunpycm.cmlist.get("sdoaia1700"),
    "171a": sunpycm.cmlist.get("sdoaia171"),
    "193a": sunpycm.cmlist.get("sdoaia193"),
    "211a": sunpycm.cmlist.get("sdoaia211"),
    "304a": sunpycm.cmlist.get("sdoaia304"),
    "335a": sunpycm.cmlist.get("sdoaia335"),
    "94a": sunpycm.cmlist.get("sdoaia94"),
    "bx": color_tables.hmi_mag_color_table(),
    "by": color_tables.hmi_mag_color_table(),
    "bz": color_tables.hmi_mag_color_table(),
}

In [ ]:
# Visualization
# Example image for visualization (from test set)
timestamps = ["2019-12-25 00:24:00"]
img_indices = [
    data_module.test_ds.aligndata.index.get_loc(pd.to_datetime(i_time)) for i_time in timestamps
]
x = data_module.test_ds[img_indices[0]][0].unsqueeze(0)
corrupted_img = x.clone()

# Use config for corrupted_channel_index, fallback to 5
corrupted_channel = cfg.experiment.get("corrupted_channel_index", 5)
corrupted_img[:, corrupted_channel, :, :, :] = 0

# Forward pass
loss, x_hat, mask = backbone(corrupted_img)
x_hat_zero_shot, mask_zero_shot = zero_shot(corrupted_img)

# Visualization
x_hat_np = x_hat.detach().cpu().numpy()
x_hat_zero_shot_np = x_hat_zero_shot.detach().cpu().numpy()

fig, axes = plt.subplots(nrows=4, ncols=9, figsize=(15, 7))

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

for i_ch, ch in enumerate(channels):

    if i_ch == corrupted_channel:
        axes[0, i_ch].set_facecolor("black")
        axes[0, i_ch].set_aspect("equal")
        axes[0, i_ch].set_xlim(0, 1)
        axes[0, i_ch].set_ylim(0, 1)
        axes[0, i_ch].plot([0, 1], [0, 1], color="red", linewidth=2)
        axes[0, i_ch].plot([0, 1], [1, 0], color="red", linewidth=2)
        axes[0, i_ch].text(
            0.5,
            -0.1,
            "missing",
            color="red",
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
        )

    axes[0, i_ch].imshow(corrupted_img[0, i_ch, 0, :, :], cmap=cms_dict[ch])
    axes[1, i_ch].imshow(x[0, i_ch, 0, :, :], cmap=cms_dict[ch])
    axes[2, i_ch].imshow(x_hat_np[0, i_ch, 0, :, :], cmap=cms_dict[ch])
    axes[3, i_ch].imshow(x_hat_zero_shot_np[0, i_ch, 0, :, :], cmap=cms_dict[ch])

    axes[0, i_ch].set_title(f"{ch}", fontsize=12, pad=10)

plt.tight_layout()
plt.savefig("DS_missing_data_img_result.pdf", dpi=300, bbox_inches="tight")